In [1]:
from setup import test, load_data
dane = load_data()

Loading data
Data loaded


Jako mój dataset wybrałem [Speed Dating Data](https://www.kaggle.com/datasets/annavictoria/speed-dating-experiment). Bazuje on na danych zebranych między 2002 i 2004 na eksperymentalnym wydaniu spotkań z cyklu randek w ciemno. Randki, w których brali udział trwały po 4 minuty, a po nich oceniali partnera w kontekście atrakcyjności, uprzejmości, inteligencji, "fajności" (fun), ambitności oraz wspólnych zainteresowań. Dataset zawiera też odpowiedzi zebrane przed i po eventach, jednak nie użyłem ich do mojej analizy. Przedstawione modele będą próbował się nauczyć, czy rozważany uczestnik na rozważanej randce da drugiej osobie "like".

# Wstęp do moich hipotez
Romantyczne zainteresowanie drugą osobą (w tym randki więc i randki w ciemno) nie jest bezpośrednio ani wyłącznie zależne od cech partnera (w tym przypadku samej atrakcyjności `attr`, fajności `fun` i wspólnych zainteresowań `shar`). Pozwoliłem sobie w tym eksperymencie postawić kilka hipotez.

In [3]:
# Domyślny, indwywidualny zestaw cech: ['attr', 'fun', 'shar']
test(dane, ['attr', 'fun', 'shar'], 'base')


ŚREDNIE WYNIKI DLA base (13 prób):
Accuracy: 75.11% | Recall: 69.15% | Fałszywe alarmy: 156.54


{'avg_accuracy': 75.11312319682195,
 'avg_recall': 69.15064102564102,
 'avg_false': 156.53846153846155}

## Hipoteza nr. 1 - Atrakcyjność względna
Wzięcie pod uwagę relatywnej atrakcyjności (w stosunku do średniej na danym spotkaniu) pozwoli modelowi trafniej przewidywać rezultat randki. Należy pamiętać, żeby odnosić jednostkę do jej "konkurencji", a nie wszystkich (czyli osób tej samej płci, na tym samym spotkaniu)

In [5]:
test(dane, ['attr', 'fun', 'shar', 'attr_rel'], 'rel')


ŚREDNIE WYNIKI DLA rel (13 prób):
Accuracy: 75.70% | Recall: 68.43% | Fałszywe alarmy: 144.46


{'avg_accuracy': 75.703076674388,
 'avg_recall': 68.42948717948718,
 'avg_false': 144.46153846153845}

### Wniosek hipoteza nr. 1
Dokładność nieznacznie wzrosła, jednak recall spadł o prawie 0.8 punktu procentowego - nasuwa się pytanie co w takim razie jest ważniejsze niż atrakcyjność ogółu?

## Hipoteza nr. 2A - Atrakcyjność poprzedniej/ego rozmówcy/czyni
Patrząc z socjologicznego punktu widzenia, ważniejsza będzie atrakcyjność ostatniego rozmówcy - jednostce ciężko jest wziąć pod uwagę wszystkich na sali. Zobaczmy, czy pozwoli nam to lepiej zgadywać decyzję (like or pass) randkującego.
Uwzględnijmy więc oba parametry

In [7]:
test(dane, ['attr', 'fun', 'shar', 'attr_rel', 'attr_prev'], 'attr_rel attr_prev')


ŚREDNIE WYNIKI DLA attr_rel attr_prev (13 prób):
Accuracy: 75.46% | Recall: 69.40% | Fałszywe alarmy: 153.31


{'avg_accuracy': 75.46251232807452,
 'avg_recall': 69.40438034188034,
 'avg_false': 153.30769230769232}

### Wniosek hipoteza nr. 2A
Tym razem udało nam się podnieść zarówno accuracy jak i recall względem zwykłego zgadywania na bazie wyłącznie atrybutów rozmówcy. Hipoteza okazała się więc być prawdziwa.

## Hipoteza nr. 2B - Atrakcyjność poprzednich rozmówców/rozmówczyń
W hipotezie 2A uwzględnialiśmy tylko poprzednią randkę. Co jeśli dać modelowi dostęp do informacji o **kilku** poprzednich randkach? Przecież chętniej damy "like" komuś, jeśli od paru randek trafialiśmy na słabych kandydatów/kandydatki. Oczywiście im dawniej one były tym mniej istotne dla nas będą, więc musimy uwzglęnić to licząc `attr_mem`

In [10]:
test(dane, ['attr', 'fun', 'shar', 'attr_rel', 'attr_memory'], 'attr_rel attr_memory')


ŚREDNIE WYNIKI DLA attr_rel attr_memory (13 prób):
Accuracy: 75.58% | Recall: 69.52% | Fałszywe alarmy: 152.46


{'avg_accuracy': 75.57706603637108,
 'avg_recall': 69.52457264957265,
 'avg_false': 152.46153846153845}

### Wniosek hipoteza nr. 2B
Jak widać udało nam się poprawić dokładność modelu w obydwóch metrykach **względem modelu, który uwzględnia tylko poprzednią randkę (2A)**.

### Hipoteza nr. 3 - Attr, fun i shar poprzednich rozmówczyń/ców
Zastosowanie takiej pamięci dla wszystkich trzech cech, które rozważamy (`attr`, `fun`,`shar`) poprawi dokładność modelu - przecież liczy się to wszystko, a nie tylko atrakcyjność.

In [11]:
test(dane, ['attr', 'fun', 'shar', 'attr_memory', 'fun_memory', 'shar_memory', 'attr_rel'], 'full memory attr_rel')


ŚREDNIE WYNIKI DLA full memory attr_rel (13 prób):
Accuracy: 76.36% | Recall: 71.61% | Fałszywe alarmy: 153.92


{'avg_accuracy': 76.3617611848391,
 'avg_recall': 71.60790598290599,
 'avg_false': 153.92307692307693}

### Wniosek hipoteza nr. 3
Ponownie, jeszcze dokładniej udało nam się przewidywać rezultaty randek.

### Hipoteza nr. 4A - Decyzja przy poprzedniej randce
Według mnie najciekawsza z perspektywy socjologicznej hipoteza. Mimo, że danie like na takim wydarzeniu nie jest wcale zobowiązujące, uważam, że będzie zaniżać szanse następnego rozmówcy, abyśmy mu również dali like. Powód przypisywałbym tutaj do uczucia podobnego do zdrady - choć na mniejszą skalę, i tak skutecznie może nas zniechęcić do kolejnego like. Warto rozważyć czy nie są to dane nakładające się z informacjami o poprzednich rozmówcach, jednak uważam, że nie - mimo wysokiej atrakcyjności itd. i tak mogliśmy komuś nie dać like.

In [12]:
test(dane, ['attr', 'fun', 'shar', 'attr_memory', 'fun_memory', 'shar_memory', 'attr_rel', 'dec_prev'], 'full memory attr_rel dec_prev')


ŚREDNIE WYNIKI DLA full memory attr_rel dec_prev (13 prób):
Accuracy: 78.59% | Recall: 73.44% | Fałszywe alarmy: 134.54


{'avg_accuracy': 78.58983874320984,
 'avg_recall': 73.4375,
 'avg_false': 134.53846153846155}

### Wniosek hipoteza nr. 4A - Znaczny skok
Jak widać okazało sie, że decyzja w trakcie poprzedniej randki miała większe znaczenie w obydwóch metrykach niż wszystkie poprzednie dane. Wyniki w takim razie okazały się wspierać hipotezę - zarówno accuracy jak i recall podniosły się odpowiednio o 2.23 oraz 1.83 punkty procentowe.

## Hipoteza nr. 4B - Decyzja partnera z poprzedniej randki
Idąc o krok dalej, możemy domniemywać, że match (czyli like z obydwóch stron) jeszcze bardziej sprawi, że zaczniemy się czuć w wyżej wspomniany sposób, a ekscytacja nim może skutecznie utrudnić powodzenie randki następnemu rozmówcy. Jednak w przeprowadzonym badaniu informację o matchu uczestnicy dostawali dopiero po wydarzeniu - mailowo. W takim razie można byłoby to w pewien sposób traktować jako data leakage, choć moim zdaniem tak nie jest - z perspektywy uczestnika traktowałbym to bardziej jako wskaźnik uznania randki za udaną z **obu** stron, co "nasza" strona czuć mogła (i uważam, że zazwyczaj czuła), więc psychicznie mogła zachowywać się jak pewnego rodzaju podwójny like. Z punktu widzenia ML, data leakage to nie jest - mimo, że użytkownik po każdej turze tej informacji jednoznacznie nie zna, system/model już tak (wiadomo, zależy od implementacji).

In [13]:
test(dane, ['attr', 'fun', 'shar', 'attr_memory', 'fun_memory', 'shar_memory', 'attr_rel', 'dec_prev','dec_o_prev'], 'full memory attr_rel dec_prev dec_o_prev')


ŚREDNIE WYNIKI DLA full memory attr_rel dec_prev dec_o_prev (13 prób):
Accuracy: 79.07% | Recall: 74.77% | Fałszywe alarmy: 135.77


{'avg_accuracy': 79.07096560184772,
 'avg_recall': 74.77297008547008,
 'avg_false': 135.76923076923077}

### Wnioski hipoteza nr. 4B
Hipoteza okazała się ponownie być trafiona. Dalej znacznie podniosła dokładność naszego modelu, jednak już nie aż tak jak decyzja jednostki, którą rozważamy.

# Podsumowanie
Randkowanie, jak większość z nas zakładam, że wie to nie tylko my i nasze atrybuty. Liczy się też nasza konkurencja, ale też nasz rozmówca. Jak pokazuje mój eksperyment (czyli dane), czasem nieoczywiste na pierwszy rzut oka dane mogą mieć znaczenie - tak jak w tym przypadku `dec_o_prev`, czyli decyzja poprzedniego rozmówcy. Dokładność modelu udało nam się podnieść z:

Accuracy: 75.11% | Recall: 69.15% | Fałszywe alarmy: 156.54
dla wersji `base`

do

Accuracy: 79.07% | Recall: 74.77% | Fałszywe alarmy: 135.77
dla wersji `full memory attr_rel dec_prev dec_o_prev`